[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-18-ibm-model-1.ipynb)

# Lecture 18: IBM Model 1, word alignment learned by counting

Before neural MT, a translation system had to know which word translates which, and no corpus
marks that. IBM Model 1 (Brown and colleagues, 1993) learned it anyway, from sentence pairs
alone, with an algorithm called EM: guess, count, guess again. A Transformer's attention also
ends up relating each word it writes to words of the source, and it never runs EM. This notebook
runs Model 1 on thirteen English-Spanish pairs, then sets its alignment of one sentence pair
beside a Transformer's attention on the same pair.

By the end you will be able to:

1. say what a word alignment is, and how EM learns translation probabilities from sentence pairs
   in which no alignment is marked;
2. explain why one round of EM cannot settle what `house` translates to, and what the later
   rounds use to settle it;
3. compare Model 1's alignment with a Transformer's cross-attention on the same sentence pair:
   where they agree, where they differ, and why attention needs no EM.

These serve Lecture 18's outcome *Explain how word alignments relate to attention in NMT*.

One prediction stops the notebook until you write it; otherwise you run the cells and read.
Everything runs on a CPU in under a minute, so no GPU is needed. About 10 minutes.

**Running in Google Colab?** Run the cells in order. The second downloads a small trained
Transformer, about 11 MB. There is nothing to uncomment.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 18 (*Research in the MATRIX Lab, Open Problems, and What Earlier MT Methods Teach*).

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
from torchlingo.colab import setup

# The trained Transformer from Lecture 8b and its subword vocabulary, used in Step 5. A pip
# install does not ship them, so setup downloads any that are missing.
env = setup(data=["data/pretrained/model.pt", "data/pretrained/spm.model"])

from collections import defaultdict

import torch

from torchlingo.visualization import format_attention

## Step 1: thirteen sentence pairs, and no alignments

A **word alignment** says which source word each target word translates. In *the red car* and
*el coche rojo* it is `el` with `the`, `coche` with `car` and `rojo` with `red`, the adjective
and the noun in the opposite order. A parallel corpus gives you the sentence pairs and never the
alignment.

The pairs below are the whole corpus. You can see at once which Spanish word goes with `house`,
because you know what the words mean. Model 1 does not: all it has is which words turn up in
the same pair.

In [ ]:
PAIRS = [
    ("the house", "la casa"),
    ("a house", "una casa"),
    ("the red house", "la casa roja"),
    ("the big house", "la casa grande"),
    ("the door", "la puerta"),
    ("a red door", "una puerta roja"),
    ("the big door", "la puerta grande"),
    ("the car", "el coche"),
    ("a car", "un coche"),
    ("the red car", "el coche rojo"),
    ("the big car", "el coche grande"),
    ("the book", "el libro"),
    ("a big book", "un libro grande"),
]

# Each pair as two lists of words.
pairs = [(english.split(), spanish.split()) for english, spanish in PAIRS]
print(f"{len(pairs)} pairs")

## Step 2: EM, guess, count, guess again

Model 1 keeps one number for each pair of words, t(s | e): the probability that English word `e`
translates as Spanish word `s`. It starts knowing nothing, every Spanish word equally likely for
every English word, and repeats one round of two steps:

1. **Expect.** In each sentence pair, every Spanish word came from one of the English words, but
   which one is hidden. So share the Spanish word out among them, in proportion to the current t:
   a word with one strong candidate gives it nearly all the credit, and a word with no favourite
   splits it evenly. Each share is a *fractional count*.
2. **Maximize.** Add up the fractional counts over the whole corpus, and for each English word
   divide by its total, so that its probabilities sum to 1. That is the new t.

The cell below is the whole algorithm, in about fifteen lines. The real Model 1 also lets a
Spanish word come from no English word at all, the NULL word; leaving it out keeps the tables
small.

In [ ]:
def em_round(t, pairs):
    """One round of EM: share each Spanish word among its pair's English words, then renormalize."""
    count = defaultdict(float)   # fractional count of each (Spanish word, English word)
    total = defaultdict(float)   # fractional count of each English word
    for english, spanish in pairs:
        for s in spanish:
            # Expect: each English word in this pair gets a share of s, in proportion to t.
            z = sum(t[s, e] for e in english)
            for e in english:
                share = t[s, e] / z
                count[s, e] += share
                total[e] += share
    # Maximize: the new t(s | e) is e's count for s, over all of e's counts.
    return {(s, e): count[s, e] / total[e] for (s, e) in t}


def start(pairs):
    """Round 0: every Spanish word equally likely for every English word."""
    english = {e for en, _ in pairs for e in en}
    spanish = {s for _, es in pairs for s in es}
    return {(s, e): 1 / len(spanish) for e in english for s in spanish}


def best(t, e, n=3):
    """The n Spanish words most likely for English word e, with their probabilities."""
    ranked = sorted(((p, s) for (s, word), p in t.items() if word == e), reverse=True)
    return "  ".join(f"{s} {p:.2f}" for p, s in ranked[:n])

## Step 3: predict, then run one round

In round 0, `house` is equally likely to be any of the Spanish words. **Write a prediction
first**, in `PREDICTION` below: after one round of EM, which Spanish word will `house` be most
likely to translate to, and why can Model 1 not be sure of it yet? The pairs in Step 1 hold the
answer. The notebook stops at the next cell until you write it.

In [ ]:
PREDICTION = ""   # <- your prediction, in a sentence or two
assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."

In [ ]:
t = start(pairs)
spanish_words = sorted({s for s, _ in t})
print(f"round 0  house: each of the {len(spanish_words)} Spanish words {1 / len(spanish_words):.2f}")

t = em_round(t, pairs)
print(f"round 1  house: {best(t, 'house')}")
print()
print("You predicted:", PREDICTION.strip())

`casa` leads, with `la` not far behind. `house` is in four pairs: `casa` is in all four, and
`la` in three of them. From one pair at a time, counting cannot tell which of the two is
`house`'s translation and which belongs to another word.

The corpus as a whole can. `la` also turns up beside `door`, where there is no house, and every
pair with `la` has `the` in it. Once `the` claims a larger share of `la`, there is less of `la`
left for `house`, and `casa` gains. Each round uses the last round's t to share the credit out
more sharply. Nine more rounds:

In [ ]:
for round_number in range(2, 11):
    t = em_round(t, pairs)
    if round_number in (2, 3, 5, 10):
        print(f"round {round_number:2d}  house: {best(t, 'house')}")

print()
print("After ten rounds, the likeliest translations of each English word:")
for e in sorted({e for _, e in t}):
    print(f"  {e:>6}: {best(t, e)}")

`house` has settled on `casa`, and every noun on its own translation. Two words have not
settled, and should not: `the` is split between `la` and `el`, and `red` between `roja` and
`rojo`, because Spanish marks gender where English does not. Model 1 learns one probability for
each pair of words and nothing about which noun is feminine, so the most it can say is that both
are translations, in proportion to how often each occurred.

## Step 4: Model 1's alignment of one sentence pair

With t learned, Model 1 can align any pair: for each Spanish word, how its credit is shared
across that pair's English words. This is the Expect step again, for one pair, and each row sums
to 1. Here is *the red car* and *el coche rojo*, in percent.

In [ ]:
def model1_alignment(t, english, spanish):
    """For each Spanish word, its share across the English words of one pair."""
    rows = []
    for s in spanish:
        z = sum(t[s, e] for e in english)
        rows.append([t[s, e] / z for e in english])
    return torch.tensor(rows)


ENGLISH, SPANISH = "the red car".split(), "el coche rojo".split()
model1 = model1_alignment(t, ENGLISH, SPANISH)
print(format_attention(model1, ENGLISH, SPANISH, show_values=True))

## Step 5: the same pair, through a Transformer's attention

A Transformer keeps no table of word pairs. Each time its decoder writes a word, **cross-attention**
gives every source word a weight, the weights summing to 1, and the decoder reads the source as
that weighted mix. So a row of weights says where the model looked while it wrote that word: the
same shape as a row of Model 1's alignment. Lecture 8b's notebook computes one attention step,
and [tutorial 4](../tutorials/04-attention-and-alignment.ipynb) measures how well attention
alignments match the right ones, if you want more.

The model is the small English-Spanish Transformer from Lecture 8b. The cell gives it *The red
car.* and the Spanish *El coche rojo.*, rather than letting it write its own translation, so that
both methods align the same pair. It records the last layer's cross-attention, averaged over the
layer's four heads. The model reads subword pieces (`coche` is `▁co` and `che`), so the cell adds
each word's pieces back together, giving one row per Spanish word and one column per English
word, and leaves out the start, end and full stop, so each row again sums to 1.

In [ ]:
# Setup: run this cell, no need to read it.
# It loads Lecture 8b's model, gives it an English sentence and a Spanish translation, and
# records the last decoder layer's cross-attention, averaged over its heads, while the model reads
# the Spanish. A Spanish word's row is the mean of its pieces' rows, an English word's column the
# sum of its pieces' columns; start, end and punctuation are dropped and each row rescaled to 1.
from torchlingo.checkpoint import load_model
from torchlingo.data_processing.vocab import SentencePieceVocab

vocab = SentencePieceVocab("data/pretrained/spm.model")
transformer, _, _ = load_model("data/pretrained/model.pt", "cpu", src_vocab=vocab)


def words_of(pieces):
    """Group piece positions into lowercase words, skipping <sos>, <eos> and punctuation."""
    words, positions, starts_word = [], [], True
    for i, piece in enumerate(pieces):
        text = piece.replace("▁", "")
        starts_word = starts_word or piece.startswith("▁")
        if piece.startswith("<") or not any(ch.isalnum() for ch in text):
            continue
        if starts_word or not words:
            words.append("")
            positions.append([])
        words[-1] += text.lower()
        positions[-1].append(i)
        starts_word = False
    return words, positions


def attention_by_word(english, spanish):
    """The last layer's cross-attention, one row per Spanish word, one column per English word."""
    source = torch.tensor([vocab.encode(english, add_special_tokens=True)])
    target = torch.tensor([vocab.encode(spanish, add_special_tokens=True)])
    cross = transformer.transformer.decoder.layers[-1].multihead_attn
    seen = {}

    def recording(query, key, value, *args, _forward=cross.forward, **kwargs):
        kwargs.update(need_weights=True, average_attn_weights=True)
        out, weights = _forward(query, key, value, *args, **kwargs)
        seen["weights"] = weights[0]
        return out, weights

    cross.forward = recording
    try:
        with torch.no_grad():
            # The decoder reads <sos> and the Spanish so far: row n is the state that writes piece n.
            transformer(source, target[:, :-1])
    finally:
        del cross.forward

    english_words, english_at = words_of(vocab.indices_to_tokens(source[0].tolist()))
    spanish_words, spanish_at = words_of(vocab.indices_to_tokens(target[0].tolist())[1:])
    weights = seen["weights"]
    by_word = torch.tensor(
        [[weights[rows][:, cols].sum(dim=1).mean().item() for cols in english_at] for rows in spanish_at]
    )
    return by_word / by_word.sum(dim=1, keepdim=True), english_words, spanish_words


attention, english_words, spanish_words = attention_by_word("The red car.", "El coche rojo.")
assert (english_words, spanish_words) == (ENGLISH, SPANISH), (english_words, spanish_words)

print("Model 1, after ten rounds of EM:")
print(format_attention(model1, ENGLISH, SPANISH, show_values=True))
print()
print("The Transformer's cross-attention, last layer, four heads averaged:")
print(format_attention(attention, ENGLISH, SPANISH, show_values=True))

## Step 6: where they agree, where they differ

Compare the two tables row by row. **Before you read on**, write in `COMPARE` one row where they
agree and one where they differ, and a guess at why they differ. Then discuss with your neighbour:
Model 1 needed ten rounds of EM to find its alignment; the Transformer never ran EM at all. How
did it get one?

In [ ]:
COMPARE = ""   # <- one row where they agree, one where they differ, and why you think they differ

print("You wrote:", COMPARE.strip() or "(nothing yet)")

### Read it

**Both put `el` on `the`, but Model 1 holds part of it back for `car`.** In this corpus `el`
never appears without `car` or `book`, so counting cannot rule `car` out. The Transformer,
trained on far more text, has no such doubt.

**On `coche` and `rojo` it is the other way round.** Model 1 is sure of each, and the attention
spreads both across *red car*. Attention is not trained to align words. It is trained, with the
rest of the model, to help predict the next word, and to write a noun and its adjective, the
helpful thing to read is the phrase, the noun with its adjective. A row of attention is where the
model found it useful to look. Often that is the aligned word, as Lecture 8b's map showed; nothing
obliges it to be.

**Why attention needs no EM.** Model 1 treats the alignment as a hidden choice: each Spanish word
came from exactly one English word, and which one is unknown. So it must guess the alignment from
t, re-estimate t from the guesses, and repeat. Attention makes no hidden choice. Its weights are
computed from the words by a function with learned parameters, and every step of that function
can be differentiated, so the training that teaches the model to translate, gradient descent on
the loss, adjusts the attention along with everything else. The alignment is never a target. It
comes out as a by-product of learning to translate, which is also why it can be fuzzier than
Model 1's, and why it can use context Model 1 never sees.

## Your turn, if there is time: add a pair

Model 1 gave part of `el` to `car` only because nothing in the corpus shows `el` without `car` or
`book`. Add a pair to `PAIRS` in Step 1 that does: `("the boy", "el niño")`, or one of your own.
Before you run anything, decide what you expect `el`'s row in Step 4 to do. Then click in the
`PAIRS` cell and choose **Runtime → Run cell and below**.

What moved, besides `el`? Look at `the` in the table after round 10. Then try a pair that
misleads the counting rather than helping it, and see how far one pair can move the table.